# MHC II IHC group DEG analysis — CosMx epithelial cells

**Goal:** Identify differentially expressed genes in CosMx epithelial (malignant) cells
between MHC II IHC-positive and IHC-negative patient groups. Wilcoxon rank-sum tests
provide an initial gene ranking; pseudobulk DESeq2 (PyDESeq2) provides the primary
statistical comparison. Results are visualized as ranked gene bar plots, a volcano
plot, and an MA plot.

**Input:** `outputs/processed/tumor_data_scored.h5ad` — CosMx AnnData with cell type
labels; `data/patient_mhc_ii_classifications.csv` — IHC-based MHC II patient groups.

**Output:** `outputs/figures/fig3d.pdf`, `fig3e_neg.pdf`, `fig3e_pos.pdf`,
`fig3g_volcano.pdf`, `fig3g_ma.pdf`; `outputs/tables/deseq2_mhc2_pos_vs_neg.csv`.

## Setup

In [ ]:
# standard libraries
from pathlib import Path
import yaml

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from adjustText import adjust_text
import scipy.sparse as sp

# single-cell analysis
import anndata as ad
import scanpy as sc

# pseudobulk differential expression
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats
from pydeseq2.default_inference import DefaultInference

# figure settings
sns.set(font_scale=1.8)
sns.set_style('ticks')
plt.rcParams['pdf.fonttype'] = 42  # embed fonts in PDF — required by most journals
plt.rcParams['ps.fonttype'] = 42

In [ ]:
# paper-wide color palettes — consistent across all analyses
# MHC II pos = orange (#FF8811), MHC II neg = purple (#462255)
cmap_low_high = ['#462255FF', '#FF8811FF', '#9DD9D2FF', '#046E8FFF', '#D44D5CFF']  # neg first
cmap_high_low = ['#FF8811FF', '#462255FF', '#9DD9D2FF', '#046E8FFF', '#D44D5CFF']  # pos first
cmap_umap     = ['#9DD9D2FF', '#462255FF', '#FF8811FF', '#046E8FFF', '#D44D5CFF']  # used in CosMx/scRNA UMAPs

# patient classification color map — used consistently across UMAP and bar plots
classification_palette = {
    'class II positive':      cmap_high_low[0],
    'class II negative':      cmap_high_low[1],
    'PT class II positive':   cmap_high_low[0],
    'PT class II negative':   cmap_high_low[1],
    'PT class II clonal':     cmap_high_low[2],
    'PT no malignant cells':  cmap_high_low[3],
    'CT class II positive':   cmap_high_low[0],
    'CT class II negative':   cmap_high_low[1],
    'CT class II clonal':     cmap_high_low[2],
    'CT no malignant cells':  cmap_high_low[3],
}

In [ ]:
# load paths from central config
# collaborators: update this path to point to your local copy of paths_config.yml
with open('/home/gh8sj/projects/mhc2-luad-paper/data/paths_config.yml') as f:
    cfg = yaml.safe_load(f)

repo_root = Path(cfg['repo_root'])

# inputs
tumor_scored_path = repo_root / cfg['outputs']['processed'] / 'tumor_data_scored.h5ad'
patient_class_path = Path(cfg['datasets']['patient_classifications']['raw'])

# output paths — organized by figure number
fig_dir   = repo_root / cfg['outputs']['figures']
table_dir = repo_root / cfg['outputs']['tables']

fig_out      = fig_dir / 'figure3'
supp_out     = fig_dir / 'figure3-supp'
table_out    = table_dir / 'figure3'
processed_out = repo_root / cfg['outputs']['processed']

fig_out.mkdir(parents=True, exist_ok=True)
supp_out.mkdir(parents=True, exist_ok=True)
table_out.mkdir(parents=True, exist_ok=True)
processed_out.mkdir(parents=True, exist_ok=True)

sc.settings.figdir = supp_out

## Load data

The scored CosMx AnnData (output of `cosmx_cell_typing`) and IHC-based patient
MHC II classifications (made in collaboration with Dr. Stelow) are loaded here.
Cell type labels are verified before subsetting to epithelial cells.

In [ ]:
# load CosMx AnnData with cell type labels assigned in cosmx_cell_typing
labeled_adata = ad.read_h5ad(tumor_scored_path)
print(labeled_adata.obs['cell_type_assigned'].value_counts())

In [ ]:
# rename cell_type_assigned to something more polished
labeled_adata.obs['Assigned Cell Type'] = labeled_adata.obs['cell_type_assigned']

In [ ]:
# load IHC-based patient MHC II classifications (positive / negative / clonal)
patient_classifications = pd.read_csv(patient_class_path, index_col=0)
patient_classifications.loc['tonsil'] = ['Tonsil', 'Tonsil', 'Tonsil']
patient_classifications['patient'] = patient_classifications.index
print(patient_classifications['patient classification'].value_counts())

## UMAP: MHC II gene expression and cell type

UMAPs colored by MHC II gene expression (HLA-DRA, HLA-DRB, HLA-DPA1, HLA-DPB1,
HLA-DQA1, HLA-DQB1/2, CD74) and cell type assignment provide an overview of MHC II
expression across the CosMx tumor microenvironment.

In [ ]:
# set the output directory

sc.settings.figdir = fig_out

mhc2_genes = ['HLA-DRA', 'HLA-DRB', 'HLA-DPA1', 'HLA-DPB1',
               'HLA-DQA1', 'HLA-DQB1/2', 'CD74', 'Assigned Cell Type']

sc.pl.umap(
    labeled_adata,
    color=mhc2_genes,
    s=1,
    cmap='viridis',
    save='_figS3a.pdf',
)



## Subset: epithelial lineage cells

Epithelial cells (the malignant compartment) are isolated from the full CosMx
object and re-embedded with their own UMAP for classification-level comparisons.
IHC patient classifications are merged into the epithelial cell metadata.

In [ ]:
%%time
# define processed_out in the paths cell alongside fig_out
processed_out = repo_root / cfg['outputs']['processed']
processed_out.mkdir(parents=True, exist_ok=True)

epithelial_path = processed_out / 'epithelial.h5ad'

if epithelial_path.exists():
    print("Loading pre-computed epithelial embedding...")
    epithelial = ad.read_h5ad(epithelial_path)
    print(f"Loaded: {epithelial.n_obs:,} cells")
else:
    # subset to epithelial lineage (malignant) cells only
    epithelial = labeled_adata[labeled_adata.obs['cell_type_assigned'] == 'Epithelial'].copy()
    print(f"Epithelial cells: {epithelial.n_obs:,}")

    # re-embed epithelial cells in their own UMAP space
    sc.pp.neighbors(epithelial)
    sc.tl.umap(epithelial)
    sc.tl.leiden(epithelial)

    # save checkpoint
    epithelial.write(epithelial_path)
    print(f"Saved → {epithelial_path}")

In [ ]:
# merge IHC patient classifications into epithelial cell metadata
epithelial.obs['patient'] = epithelial.obs['patient'].astype(str)
patient_classifications['patient'] = patient_classifications['patient'].astype(str)

epithelial.obs = pd.merge(
    epithelial.obs, patient_classifications, on='patient', how='left'
)

# create a unique per-core identifier (patient × region)
epithelial.obs['patient_region'] = (
    epithelial.obs['patient'].astype(str) + '_' + epithelial.obs['region'].astype(str)
)

print('Patients by classification:\n', patient_classifications['patient classification'].value_counts())
print('\nEpithelial cells by classification:\n', epithelial.obs['patient classification'].value_counts())

In [ ]:
epithelial_classified = epithelial[
    epithelial.obs['patient classification'].isin(['class II positive', 'class II negative'])
].copy()

sc.pl.umap(
    epithelial_classified,
    color='patient classification',
    palette=classification_palette,
    s=3,
    save='_figS3d.pdf',
)

## UMAP: epithelial cells by IHC classification

Epithelial cell UMAPs colored by MHC II gene expression and patient IHC
classification. The MHC II positive vs negative comparison is the primary
figure panel (fig3d). Peripheral tumor (PT) and central tumor (CT) region
breakdowns are shown as supplementary views.

In [ ]:
sc.settings.figdir = fig_out

# MHC II gene expression on epithelial UMAP — fig3d (main figure)

sc.pl.umap(
    epithelial,
    color=['HLA-DRA', 'HLA-DRB', 'HLA-DPA1', 'HLA-DPB1',
           'HLA-DQA1', 'HLA-DQB1/2', 'CD74', 'cell_type_assigned'],
    s=3, cmap='viridis', palette='Set1',
    save='_fig3d.pdf',
)

# MHC II pos vs neg — fig3d supplemental
sc.settings.figdir = supp_out
sc.pl.umap(
    epithelial_classified,
    color='patient classification',
    palette=classification_palette,
    s=3,
    save='_figS3d.pdf',
)

# reset to supp_out as default for remaining cells
sc.settings.figdir = supp_out

In [ ]:
# peripheral tumor (PT) samples
print('PT patients by group:\n', patient_classifications['pt sample classification'].value_counts())
print('\nPT epithelial cells by group:\n',
      epithelial[epithelial.obs.region == 'PT'].obs['pt sample classification'].value_counts())

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sc.pl.umap(epithelial[epithelial.obs.region == 'PT'],
           color='pt sample classification',
           palette=classification_palette,
           s=3, ax=axes[0], show=False, legend_loc='right margin')
sc.pl.umap(epithelial[epithelial.obs.region == 'PT'],
           color='patient',
           s=3, ax=axes[1], show=False, legend_loc='none')
plt.tight_layout()
plt.show()

# central tumor (CT) samples
print('CT patients by group:\n', patient_classifications['ct sample classification'].value_counts())
print('\nCT epithelial cells by group:\n',
      epithelial[epithelial.obs.region == 'CT'].obs['ct sample classification'].value_counts())

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sc.pl.umap(epithelial[epithelial.obs.region == 'CT'],
           color='ct sample classification',
           palette=classification_palette,
           s=3, ax=axes[0], show=False, legend_loc='right margin')
sc.pl.umap(epithelial[epithelial.obs.region == 'CT'],
           color='patient',
           s=3, ax=axes[1], show=False, legend_loc='none')
plt.tight_layout()
plt.show()

## Wilcoxon DEG: class II positive vs negative

Wilcoxon rank-sum test identifies genes enriched in each IHC group within the
epithelial cell compartment. The top 50 genes per group are plotted as ranked
bar charts. S100P co-expression with MHC II genes is also explored via Pearson
correlation across all epithelial cells.

In [ ]:
# note: two separate rank_genes_groups calls are needed because scanpy stores
# only the queried groups, not the reference. Using key_added keeps both results
# in adata.uns without overwriting each other.

sc.tl.rank_genes_groups(
    epithelial,
    groupby='patient classification',
    groups=['class II positive'],
    reference='class II negative',
    method='wilcoxon',
    key_added='rank_genes_pos',
)

sc.tl.rank_genes_groups(
    epithelial,
    groupby='patient classification',
    groups=['class II negative'],
    reference='class II positive',
    method='wilcoxon',
    key_added='rank_genes_neg',
)

In [ ]:
def plot_top_genes(adata, group, key, color, title, save_path, n=50):
    """Bar plot of top n Wilcoxon-ranked genes for a given group."""
    rank_df = sc.get.rank_genes_groups_df(adata, group=group, key=key)
    if 'feature_name' in adata.var.columns:
        gene_map = adata.var['feature_name'].to_dict()
        rank_df['gene_label'] = rank_df['names'].map(gene_map).fillna(rank_df['names'])
    else:
        rank_df['gene_label'] = rank_df['names']

    top = rank_df.sort_values('scores', ascending=False).head(n)

    fig, ax = plt.subplots(figsize=(8, 12))
    sns.barplot(y='gene_label', x='scores', data=top, color=color, fill=False, ax=ax)
    ax.set_ylabel('Gene', fontsize=14)
    ax.set_xlabel('Score', fontsize=14)
    ax.tick_params(labelsize=12)
    ax.set_title(title, fontsize=13)
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    plt.savefig(save_path)
    plt.show()

In [ ]:
sc.settings.figdir = supp_out

plot_top_genes(
    epithelial, group='class II negative', key='rank_genes_neg',
    color=cmap_high_low[1],
    title='Top 50 enriched genes — MHC II negative malignant cells (CosMx)',
    save_path=supp_out / 'figS3f.pdf',
)

plot_top_genes(
    epithelial, group='class II positive', key='rank_genes_pos',
    color=cmap_high_low[0],
    title='Top 50 enriched genes — MHC II positive malignant cells (CosMx)',
    save_path=supp_out / 'figS3e.pdf',
)

In [ ]:
# Pearson correlation of all genes with S100P across epithelial cells
# use layers['counts'] if X has been normalized, to work with the stored matrix
import scipy.sparse as sp

X = epithelial.X
if sp.issparse(X):
    X = X.toarray()

expr_df = pd.DataFrame(X, index=epithelial.obs_names, columns=epithelial.var_names)

if 'S100P' in expr_df.columns:
    corr_df = expr_df.corrwith(expr_df['S100P'], axis=0).to_frame(name='correlation')
    corr_df = corr_df.drop(index='S100P')
    print('Top correlated with S100P:\n', corr_df.nlargest(20, 'correlation'))
    print('\nTop anti-correlated with S100P:\n', corr_df.nsmallest(20, 'correlation'))
else:
    print('S100P not found in gene set')

## Heatmaps and dot plots

Expression heatmaps and dot plots of curated MHC II marker genes confirm the
transcriptional differences between IHC classification groups in the epithelial
compartment.

In [ ]:
# set publication colors for patient classification groups
epithelial_classified.uns['patient classification_colors'] = [
    classification_palette['class II negative'],
    classification_palette['class II positive'],
]

# define marker genes 
# curated marker genes from top Wilcoxon results per group
marker_genes_dict = {
    'MHC class II Low':  ['S100P', 'MIF', 'GSTP1', 'RPL37', 'KRT18',
                          'TPI1', 'PPIA', 'CD63', 'ITM2B', 'KRT8'],
    'MHC class II High': ['CD74', 'HLA-DRB', 'HLA-DRA', 'PIGR', 'MHC I',
                          'HLA-DPA1', 'SCGB3A1', 'HLA-DQB1/2', 'IFI6', 'SLPI'],
}

sc.tl.dendrogram(epithelial_classified, groupby='patient classification')
sc.pl.heatmap(
    epithelial_classified, marker_genes_dict,
    groupby='patient classification',
    vcenter=0, vmax=2, cmap='vlag',
    dendrogram=True, swap_axes=True, figsize=(11, 8),
    save='_figS3h.pdf',
)

In [ ]:
sc.pl.dotplot(
    epithelial_classified, marker_genes_dict,
    groupby='patient classification',
    show=False,
)
fig = plt.gcf()
fig.set_size_inches(16, 4)
plt.subplots_adjust(right=0.75)
plt.savefig(supp_out / 'figS3i.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# T cell marker validation — CosMx panel limitation
# CD3/CD4/CD8 probes show near-zero expression across all cell types,
# indicating poor T cell detection in this CosMx panel. Shown as a
# panel validation note rather than a biological finding.

gene_dict = {
    'T Cells': ['CD3D', 'CD3E', 'CD3G', 'CD4', 'CD8A', 'TNFRSF4'],
    'Other':   ['CXCL17', 'MIF'],
}

sc.pl.dotplot(
    labeled_adata,
    gene_dict,
    groupby='Assigned Cell Type',
    dendrogram=False,
    show=False,
)
fig = plt.gcf()
fig.set_size_inches(10, 4)
plt.subplots_adjust(right=0.75)
plt.savefig(supp_out / 'figS3_tcell_panel_validation.pdf', bbox_inches='tight')
plt.show()

## Pseudobulk DESeq2: class II positive vs negative

Epithelial cells are pseudobulked by patient (summed raw counts per patient)
before running PyDESeq2. This pseudobulk approach accounts for patient-level
variation and provides more robust differential expression statistics than
per-cell tests. Only MHC II positive and negative patients are included;
clonal patients are excluded from this comparison.

In [ ]:
%%time

import scipy.sparse as sp

# pseudobulk: sum raw counts per patient across epithelial pos/neg cells
epithelial_pos_neg = epithelial[
    epithelial.obs['patient classification'].isin(['class II positive', 'class II negative'])
].copy()

# extract raw counts — use scipy to handle sparse matrix correctly
X = epithelial_pos_neg.layers['counts']
if sp.issparse(X):
    X = X.toarray()

raw_counts = pd.DataFrame(
    X,
    index=epithelial_pos_neg.obs_names,
    columns=epithelial_pos_neg.var_names,
)

meta = epithelial_pos_neg.obs[['patient', 'patient classification']].copy()
meta = meta.dropna(subset=['patient', 'patient classification'])
raw_counts = raw_counts.loc[meta.index]

# sum counts per patient (pseudobulk)
pseudobulk_counts = raw_counts.groupby(meta['patient']).sum().astype(int)

# matched metadata: one row per patient
pseudobulk_metadata = (
    meta[['patient', 'patient classification']]
    .drop_duplicates()
    .set_index('patient')
    .rename(columns={'patient classification': 'patient_classification'})
)
pseudobulk_metadata = pseudobulk_metadata.loc[pseudobulk_counts.index]

print(f"Pseudobulked: {pseudobulk_counts.shape[0]} patients × {pseudobulk_counts.shape[1]} genes")
print(pseudobulk_metadata['patient_classification'].value_counts())

## S100P and HLA-DRA are mutually exclusive at the patient level

A striking pattern emerges when pseudobulked transcript counts are plotted per
patient: S100P and HLA-DRA expression are largely mutually exclusive in malignant
cells. Patients with high HLA-DRA tend to have low S100P, and patients with high
S100P tend to have low HLA-DRA — consistent with the IHC-based classification of
MHC II positive and negative tumors.

This pattern replicates across multiple data modalities in this study (bulk RNA-seq,
scRNA-seq, and now CosMx spatial transcriptomics), suggesting a fundamental
transcriptional dichotomy in LUAD tumor cell identity rather than a technical
artifact. The CosMx data is particularly compelling here because it confirms the
pattern at single-cell resolution within the tumor microenvironment.

The MHC II positive patients (orange) cluster in the upper-left — high HLA-DRA,
low S100P. The MHC II negative patients (purple) spread rightward — high S100P,
low or absent HLA-DRA. This mutual exclusivity is the central biological finding
of the paper, here validated spatially.

In [ ]:
# S100P vs HLA-DRA scatter colored by patient classification
plot_meta = pseudobulk_metadata.copy()
plot_meta['S100P'] = pseudobulk_counts['S100P']
plot_meta['HLA-DRA'] = pseudobulk_counts['HLA-DRA']

fig, ax = plt.subplots(figsize=(7, 6))
for group, color in [('class II negative', cmap_high_low[1]),
                     ('class II positive', cmap_high_low[0])]:
    subset = plot_meta[plot_meta['patient_classification'] == group]
    ax.scatter(subset['S100P'], subset['HLA-DRA'],
               color=color, label=group, alpha=0.8, s=40)

ax.set_xlabel('S100P')
ax.set_ylabel('HLA-DRA')
ax.set_title('HLA-DRA vs S100P transcript counts per patient — malignant cells (CosMx)')
ax.legend()
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(supp_out / 'figS3_s100p_hladra_scatter.pdf')
plt.show()

In [ ]:
%%time

inference = DefaultInference(n_cpus=6)

dds = DeseqDataSet(
    counts=pseudobulk_counts,
    metadata=pseudobulk_metadata,
    design='~patient_classification',
    refit_cooks=True,
    inference=inference,
)
dds.deseq2()

# extract results: class II positive vs negative
stat_res = DeseqStats(
    dds, inference=inference,
    contrast=['patient_classification', 'class II positive', 'class II negative'],
)
stat_res.summary()
res_deseq2 = stat_res.results_df

# preview top hits
res_deseq2.sort_values('padj').head(20)

In [ ]:
# save full DESeq2 results table
res_deseq2.to_csv(table_out / 'cosmx_deseq2_mhc2_pos_vs_neg.csv')
print(f"Saved DESeq2 results → {table_out / 'cosmx_deseq2_mhc2_pos_vs_neg.csv'}")

## Volcano plot

Volcano plot of DESeq2 results (log2 fold change vs −log10 p-value).
Genes are colored by significance and direction. KIT is highlighted
given prior reports of its association with MHC II expression in LUAD.

In [ ]:
df = res_deseq2.copy()
df['-log10(pval)'] = -np.log10(df['pvalue'])

pval_threshold = 0.05
lfc_threshold  = 1.0

df['significance'] = 'Not Significant'
df.loc[(df['pvalue'] < pval_threshold) & (df['log2FoldChange'] >  lfc_threshold), 'significance'] = 'Enriched in class II positive'
df.loc[(df['pvalue'] < pval_threshold) & (df['log2FoldChange'] < -lfc_threshold), 'significance'] = 'Enriched in class II negative'

colors = {
    'Enriched in class II positive': cmap_high_low[0],
    'Enriched in class II negative': cmap_high_low[1],
    'Not Significant': 'gray',
}

plt.figure(figsize=(16, 8))
for category, color in colors.items():
    subset = df[df['significance'] == category]
    plt.scatter(subset['log2FoldChange'], subset['-log10(pval)'],
                label=category, color=color, alpha=0.6)

# label top 50 significant genes by absolute fold change
top_genes = df[df['significance'] != 'Not Significant'].nlargest(50, 'log2FoldChange')
for gene, row in top_genes.iterrows():
    plt.text(row['log2FoldChange'], row['-log10(pval)'], gene, fontsize=10, ha='right')

plt.axhline(-np.log10(pval_threshold), linestyle='--', color='black', alpha=0.7)
plt.axvline( lfc_threshold, linestyle='--', color='black', alpha=0.7)
plt.axvline(-lfc_threshold, linestyle='--', color='black', alpha=0.7)
plt.xlabel('Log2 Fold Change')
plt.ylabel('-Log10(p-value)')
plt.title('DESeq2 volcano — MHC II positive vs negative (CosMx epithelial cells)')
plt.legend(loc='upper left', bbox_to_anchor=(1.05, 1))
plt.tight_layout()
plt.savefig(fig_out / 'fig3g_volcano.pdf')
plt.show()

## MA plot

MA plot showing mean expression (baseMean) vs log2 fold change. Genes meeting
significance and fold change thresholds are colored; labels use `adjustText`
to avoid overlap. X-axis is log-scaled.

In [ ]:
plt.figure(figsize=(10, 8))

# all genes in gray
plt.scatter(res_deseq2['baseMean'], res_deseq2['log2FoldChange'],
            c='gray', alpha=0.6, s=5)

# significant genes colored by direction
sig  = res_deseq2[res_deseq2['padj'] < 0.05]
up   = sig[sig['log2FoldChange'] >  0.85]
down = sig[sig['log2FoldChange'] < -0.85]
plt.scatter(up['baseMean'],   up['log2FoldChange'],   c=cmap_high_low[0], alpha=0.7, s=10)
plt.scatter(down['baseMean'], down['log2FoldChange'], c=cmap_high_low[1], alpha=0.7, s=10)

# label genes — no adjust_text for now to confirm plot renders
label_genes = res_deseq2[
    (res_deseq2['baseMean'] > 10**1.8) & (res_deseq2['log2FoldChange'].abs() > 0.85)
]
for gene, row in label_genes.iterrows():
    plt.text(row['baseMean'], row['log2FoldChange'], gene, fontsize=10)

plt.xscale('log')
plt.axhline(0,     linestyle='--', color='black', alpha=0.7)
plt.axhline( 0.85, linestyle='--', color='black', alpha=0.7)
plt.axhline(-0.85, linestyle='--', color='black', alpha=0.7)
plt.xlabel('Mean Expression (baseMean)')
plt.ylabel('Log2 Fold Change')
plt.title('MA plot — MHC II positive vs negative (CosMx epithelial cells)')
plt.savefig(supp_out / 'figS3k_ma.pdf', bbox_inches='tight')
plt.show()